In [ ]:
# Specify Batch

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import re, glob, os

plt.rcParams.update({"text.usetex": True})
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Times New Roman'
plt.rcParams['mathtext.fallback'] = 'stix'
plt.rcParams['font.family'] = 'Times New Roman'
plt.style.use('seaborn-v0_8-deep')
prop_cycle = plt.rcParams['axes.prop_cycle']
dcolors = prop_cycle.by_key()['color']

batch_name = "L32_OBC_m200"

base_path = rf"C:\Users\liamc\Documents\majorana_chain_dynamics\results\{batch_name}"
csv_files = glob.glob(os.path.join(base_path, "**", "*.csv"), recursive=True)
L = int(re.search(r"L(\d+)", batch_name).group(1))

def extract_params(fname):
    g_match = re.search(r"g([\d.]+)", fname)
    p_match = re.search(r"parity(\d+)", fname)
    g = float(g_match.group(1)) if g_match else None
    parity = int(p_match.group(1)) if p_match else None
    return g, parity

run_index = []

for f in csv_files:
    fname = os.path.basename(f)
    g, parity = extract_params(fname)
    if g is None:
        continue
    run_index.append({
        "path": f,
        "g": g,
        "parity": parity
    })

run_index = pd.DataFrame(run_index)

In [ ]:
# Specify Correlation Data

g = 0.5
parity = 1
i_ref = 15

site_label = i_ref + 1
j_sites = np.arange(1, L+1)
mask = j_sites != site_label
j_plot = j_sites[mask]

row = run_index[
    (run_index["g"] == g) &
    (run_index["parity"] == parity)
].iloc[0]

fname = row["path"]
print(f"Loaded: {fname}")
df = pd.read_csv(fname)

def parse_julia_array(julia_str):
    if not isinstance(julia_str, str):
        return np.array(julia_str)
    julia_str = re.sub(r'ComplexF64\[', '[', julia_str)
    julia_str = re.sub(r'(\d+)im', r'\1j', julia_str)
    return np.array(eval(julia_str))

def unflatten_correlation(flat_vec_str):
    flat_vec = parse_julia_array(flat_vec_str)
    return np.real(flat_vec.reshape(L, L))

XXC_1 = unflatten_correlation(df["XXC"].iloc[0])
XXC_2 = unflatten_correlation(df["XXC"].iloc[1])
XXCC_1 = unflatten_correlation(df["XXCC"].iloc[0])
XXCC_2 = unflatten_correlation(df["XXCC"].iloc[1])

YYC_1 = unflatten_correlation(df["YYC"].iloc[0])
YYC_2 = unflatten_correlation(df["YYC"].iloc[1])
YYCC_1 = unflatten_correlation(df["YYCC"].iloc[0])
YYCC_2 = unflatten_correlation(df["YYCC"].iloc[1])

ZZC_1 = unflatten_correlation(df["ZZC"].iloc[0])
ZZC_2 = unflatten_correlation(df["ZZC"].iloc[1])
ZZCC_1 = unflatten_correlation(df["ZZCC"].iloc[0])
ZZCC_2 = unflatten_correlation(df["ZZCC"].iloc[1])

E1 = df["energy"].iloc[0]
E2 = df["energy"].iloc[1]
gap = df["gap"].iloc[0]

print(f"\nEnergy 1 = {E1:.6f}")
print(f"Energy 2 = {E2:.6f}")
print(f"Gap = {gap:.6f}")

In [ ]:
# Correlations

fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=150)

operators = [('X', XXC_1, XXC_2), ('Y', YYC_1, YYC_2), ('Z', ZZC_1, ZZC_2)]
for ax, (op, corr1, corr2) in zip(axes, operators):
    ax.plot(j_plot, corr1[i_ref, mask], 'o-', label="State 1", color=dcolors[0], linewidth=2)
    ax.plot(j_plot, corr2[i_ref, mask], 's-', label="State 2", color=dcolors[1], linewidth=2)
    ax.set_xlabel(r"Site $j$", fontsize=12)
    ax.set_ylabel(rf"$\langle {op}_{{{site_label}}} {op}_j \rangle$", fontsize=12)
    ax.set_title(f"{op}-{op} Correlations", fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)

plt.suptitle(f"Correlations $i={site_label}$, $g/t={g}$, $L={L}$", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Connected Correlations

fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=150)

operators = [('X', XXCC_1, XXCC_2), ('Y', YYCC_1, YYCC_2), ('Z', ZZCC_1, ZZCC_2)]
for ax, (op, corr1, corr2) in zip(axes, operators):
    ax.plot(j_plot, corr1[i_ref, mask], 'o-', label="State 1", color=dcolors[0], linewidth=2)
    ax.plot(j_plot, corr2[i_ref, mask], 's-', label="State 2", color=dcolors[1], linewidth=2)
    ax.set_xlabel(r"Site $j$", fontsize=12)
    ax.set_ylabel(rf"$\langle {op}_{{{site_label}}} {op}_j \rangle - \langle {op}_{{{site_label}}} \rangle \langle {op}_j \rangle$", fontsize=12)
    ax.set_title(f"{op}-{op} Connected Correlations", fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)
    
    # ax.plot(np.sin((j_plot-1)*np.pi/L), np.abs(corr1[i_ref, mask]), 'o-', label="State 1", color=dcolors[0], linewidth=2)
    # ax.plot(np.sin((j_plot-1)*np.pi/L), np.abs(corr2[i_ref, mask]), 's-', label="State 2", color=dcolors[1], linewidth=2)
    # ax.set_xscale('log')
    # ax.set_yscale('log')
    # ax.set_xlabel(r"$\sin(\pi j / L)$", fontsize=12)
    # ax.set_ylabel(rf"$\langle {op}_{{{site_label}}} {op}_j \rangle - \langle {op}_{{{site_label}}} \rangle \langle {op}_j \rangle$", fontsize=12)
    # ax.set_title(f"{op}-{op} Connected Correlations", fontsize=12)
    # ax.grid(True, alpha=0.3)
    # ax.legend(fontsize=10)

plt.suptitle(f"Connected Correlations $i={site_label}$, $g/t={g}$, $L={L}$", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Comparison

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

operators = ['X', 'Y', 'Z']
corr_raw = [XXC_1, YYC_1, ZZC_1]
corr_cc = [XXCC_1, YYCC_1, ZZCC_1]
markers = ['o-', 's-', '^-']

for op, corr, marker in zip(operators, corr_raw, markers):
    axes[0].plot(j_plot, corr[i_ref, mask], marker, label=op, color=dcolors[operators.index(op)], linewidth=2)

for op, corr, marker in zip(operators, corr_cc, markers):
    axes[1].plot(j_plot, corr[i_ref, mask], marker, label=op, color=dcolors[operators.index(op)], linewidth=2)

axes[0].set_xlabel(r"Site $j$", fontsize=12)
axes[0].set_ylabel(r"Correlation", fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=11)

axes[1].set_xlabel(r"Site $j$", fontsize=12)
axes[1].set_ylabel(r"Connected Correlation", fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=11)

plt.suptitle(f"Comparison $i={site_label}$, $g/t={g}$, $L={L}$", fontsize=13, y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Energy Gaps

g_vals = run_index["g"].values
gaps = []

for fpath in run_index["path"]:
    df_tmp = pd.read_csv(fpath)
    gaps.append(df_tmp["gap"].iloc[0])

g_vals = np.array(g_vals)
gaps = np.array(gaps)

order = np.argsort(g_vals)
g_vals = g_vals[order]
gaps = np.array(gaps)[order]

fig, ax = plt.subplots(1, 1, figsize=(7, 5), dpi=150)
ax.plot(g_vals, gaps, 'o-', color=dcolors[0], linewidth=2, markersize=5)
ax.set_xlabel(r"$g/t$", fontsize=12)
ax.set_ylabel("Energy Gap", fontsize=12)
ax.set_title(rf"$L = {L}$", fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()